In [10]:
import os
import sys
sys.path.insert(0, '.')
import clickhouse_connect
import pandas as pd
import tqdm
from icecream import ic
from pathlib import Path
# from dotenv import load_dotenv
# from utility.utils import timer
# load_dotenv()
# PROJECT_DROPBOX_PATH = os.getenv("PROJECT_DROPBOX_PATH")
PROJECT_DROPBOX_PATH = "D:/fenghe/dropbox/Dropbox/LMSW (Diversity Washing Through the Boardroom)/1. Data"
print(PROJECT_DROPBOX_PATH)



# functions
def connect_to_clickhouse():
    client = clickhouse_connect.get_client(host='192.168.204.128', 
                                           port=8123, 
                                           username='default', 
                                           password='pm19951014',
                                           connect_timeout=600,
                                           send_receive_timeout=600)
    client.command("USE revelio071625")
    return client

#################### establish Clickhouse connection ####################

client = connect_to_clickhouse()

# Functions to execute commands by always connecting to Clickhouse first
def run_query_df(query):
    client = connect_to_clickhouse()
    return client.query_df(query)

D:/fenghe/dropbox/Dropbox/LMSW (Diversity Washing Through the Boardroom)/1. Data


# check the unique value of roles

In [ ]:
def check_data():
    query = f"""
    SELECT rcid, role_k50_v3, role_k150_v3, role_k500_v3,
       role_k1000_v3, role_k1500_v3, role_k5000_v3, role_k10000_v3,
       role_k15000_v3, onet_title, title_raw
    FROM temp_processed_global_position a
    """
    return run_query_df(query)
df = check_data()

In [17]:
# for long words such as "carbon emission", 
# we do concat('%', k.keyword, '%')
def check_data():
    query = f"""
    SELECT rcid, role_k50_v3, role_k150_v3, role_k500_v3,
       role_k1000_v3, role_k1500_v3, role_k5000_v3, role_k10000_v3,
       role_k15000_v3, onet_title, title_raw
    FROM temp_processed_global_position a
    """
    return run_query_df(query)
df = check_data()

In [18]:
df.columns

Index(['rcid', 'role_k50_v3', 'role_k150_v3', 'role_k500_v3', 'role_k1000_v3',
       'role_k1500_v3', 'role_k5000_v3', 'role_k10000_v3', 'role_k15000_v3',
       'onet_title', 'title_raw'],
      dtype='object')

In [20]:
role_columns = ['role_k50_v3', 'role_k150_v3', 'role_k500_v3',
                'role_k1000_v3', 'role_k1500_v3', 'role_k5000_v3', 
                'role_k10000_v3', 'role_k15000_v3']

with open('unique_roles_output.txt', 'w', encoding='utf-8') as f:
    for col in role_columns:
        unique_vals = sorted(df[col].dropna().unique())
        values_str = ', '.join([f'"{val}"' for val in unique_vals])
        f.write(f'{col} ({len(unique_vals)} values): {values_str}\n\n')

print("Output saved to unique_roles_output.txt")

Output saved to unique_roles_output.txt


In [11]:
def pull_unique_role_combinations():
    query = """
    SELECT DISTINCT role_k1500_v3, role_k5000_v3, role_k10000_v3
    FROM temp_processed_global_position
    WHERE role_k1500_v3 IS NOT NULL
      AND role_k5000_v3 IS NOT NULL
      AND role_k10000_v3 IS NOT NULL
    ORDER BY role_k1500_v3, role_k5000_v3, role_k10000_v3
    """
    return run_query_df(query)

df_combinations = pull_unique_role_combinations()
print(f"Total unique combinations: {len(df_combinations):,}")
print(df_combinations.head())

df_combinations.to_csv("role_combinations.csv", index=False)

Total unique combinations: 10,003
    role_k1500_v3   role_k5000_v3             role_k10000_v3
0                                                           
1  .NET Developer  .NET Developer             .NET Developer
2  .NET Developer  .NET Developer            Azure Developer
3  .NET Developer  .NET Developer           Blazor Developer
4  .NET Developer  .NET Developer  Microsoft Cloud Developer
